In [1]:
import os
import datetime
import logging
import pandas as pd
from typing import Dict, Any, Optional, List
from tqdm.auto import tqdm
from dotenv import load_dotenv
from serpapi import GoogleSearch
from simple_salesforce import Salesforce

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

# Load environment variables
load_dotenv(override=True)

# Configuration & Validation
SF_USERNAME = os.getenv('SF_USERNAME')
SF_PASSWORD = os.getenv('SF_PASSWORD')
SF_TOKEN = os.getenv('SF_TOKEN')
SERP_API = os.getenv('SERP_API')

required = ['SF_USERNAME', 'SF_PASSWORD', 'SF_TOKEN', 'SERP_API']
missing = [v for v in required if not os.getenv(v)]
if missing:
    raise ValueError(f"Missing environment variables: {', '.join(missing)}")
logger.info("Configuration loaded successfully")

/Users/tarekadammustafa/Github/Python/lead enrichment automation/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
INFO: Configuration loaded successfully


In [2]:
def extract_store_type(place: Dict[str, Any]) -> Optional[str]:
    types = place.get("type", [])
    return ", ".join(types) if isinstance(types, list) else None

def extract_service_options(place: Dict[str, Any]) -> Optional[str]:
    options = place.get("service_options", {})
    return ", ".join([k for k, v in options.items() if v]) if isinstance(options, dict) else None

def extract_payment_options(place: Dict[str, Any]) -> Optional[str]:
    for item in place.get("extensions", []):
        if "payments" in item:
            return ", ".join(item["payments"])
    return None

def extract_address_components(place: Dict[str, Any]) -> Dict[str, Optional[str]]:
    """Parse address components from full address string with improved robustness."""
    full_address = place.get("address", "")
    res = {"street": None, "city": None, "postal_code": None, "country": None}
    if not full_address: return res
        
    parts = [p.strip() for p in full_address.split(",")]
    if len(parts) >= 1: res["street"] = parts[0]
    if len(parts) >= 2:
        res["country"] = parts[-1]
        city_postal_part = parts[-2] if len(parts) > 2 else parts[1]
        import re
        postal_match = re.search(r'\b\d{4,5}(?:[-\s][A-Z]{1,2})?\b', city_postal_part)
        if postal_match:
            res["postal_code"] = postal_match.group()
            res["city"] = city_postal_part.replace(res["postal_code"], "").strip().strip(',')
        else:
            res["city"] = city_postal_part
    return res

def enrich_with_serpapi(place_id: Optional[str] = None, name: Optional[str] = None) -> Optional[Dict[str, Any]]:
    """Enrich lead data using SerpAPI Google Maps search."""
    if not place_id and not name: raise ValueError("Either place_id or name must be provided")
    params = {"engine": "google_maps", "api_key": SERP_API}
    if place_id: params["place_id"] = place_id
    else: params["q"] = name
    try:
        results = GoogleSearch(params).get_dict()
        place = results.get("place_results", {})
        if not place:
            logger.warning(f"No results for {place_id or name}")
            return None
        addr = extract_address_components(place)
        return {
            "phone": place.get("phone"),
            "website": place.get("website"),
            "rating": place.get("rating"),
            "reviews": place.get("reviews"),
            "price_range": place.get("price"),
            "store_type": extract_store_type(place),
            "service_options": extract_service_options(place),
            "payment_options": extract_payment_options(place),
            **addr,
            "full_address": place.get("address"),
        }
    except Exception as e:
        logger.error(f"Error enriching {place_id or name}: {e}")
        return None

In [3]:
# Connect to Salesforce and Query Leads
try:
    sf = Salesforce(username=SF_USERNAME, password=SF_PASSWORD, security_token=SF_TOKEN)
    logger.info("Connected to Salesforce")
    query = """
    SELECT Id, Company, Name, Phone, Website, Street, City, PostalCode, Country, 
           Google_Place_ID__c, Store_Type__c, Price_Range__c, Google_Rating__c, 
           Service_Options__c, Payment_Options__c
    FROM Lead
    WHERE Google_Place_ID__c != NULL 
    AND Lead_Enriched__c = FALSE
    AND (
        Website = NULL OR Store_Type__c = NULL OR Price_Range__c = NULL OR 
        Google_Rating__c = NULL OR Phone = NULL OR Street = NULL OR 
        City = NULL OR Country = NULL
    )
    LIMIT 10
    """
    leads = sf.query(query)['records']
    df = pd.DataFrame(leads).drop(columns=['attributes'], errors='ignore')
    logger.info(f"Retrieved {len(df)} leads to enrich")
    if not df.empty: print(df.head())
except Exception as e:
    logger.error(f"Salesforce operation failed: {e}")
    raise

INFO: Connected to Salesforce
INFO: Retrieved 1 leads to enrich


                   Id   Company           Name            Phone  \
0  00Qg50000038TzBEAU  De Beren  Tarek Mustafa  +31 36 530 3555   

                       Website     Street    City PostalCode      Country  \
0  http://www.beren.nl/welkom/  Forum 101  Almere    1315 TD  Netherlands   

            Google_Place_ID__c Store_Type__c Price_Range__c Google_Rating__c  \
0  ChIJM8l8tuYWxkcRaWuEM5ZVa_w    Restaurant              $             None   

  Service_Options__c Payment_Options__c  
0            dine_in               None  


In [4]:
# Process Enrichment with Merge Logic
preview_rows = []
for _, row in tqdm(df.iterrows(), total=len(df), desc="Enriching Leads"):
    data = enrich_with_serpapi(place_id=row.get("Google_Place_ID__c"), name=row.get("Name"))
    if data:
        # Merge logic: Only fill fields that are empty in Salesforce
        merged = {"Id": row["Id"], "Name": row["Name"]}
        field_map = {
            "Phone": "phone", "Website": "website", "Store_Type__c": "store_type",
            "Price_Range__c": "price_range", "Google_Rating__c": "rating", 
            "Google_Reviews__c": "reviews", "Service_Options__c": "service_options",
            "Payment_Options__c": "payment_options", "Street": "street", 
            "City": "city", "PostalCode": "postal_code", "Country": "country"
        }
        for sf_field, google_key in field_map.items():
            val = row.get(sf_field)
            if pd.isna(val) or not val:
                merged[sf_field] = data.get(google_key)
            else:
                merged[sf_field] = val # Keep original
        merged.update({"Lead_Enriched__c": True, "Date_Enriched_At__c": datetime.datetime.now().isoformat()})
        preview_rows.append(merged)
preview_df = pd.DataFrame(preview_rows)
logger.info(f"Successfully enriched {len(preview_df)} leads")
if not preview_df.empty: print(preview_df.head())

Enriching Leads: 100%|██████████| 1/1 [00:00<00:00, 13.93it/s]
INFO: Successfully enriched 1 leads


                   Id           Name            Phone  \
0  00Qg50000038TzBEAU  Tarek Mustafa  +31 36 530 3555   

                       Website Store_Type__c Price_Range__c  Google_Rating__c  \
0  http://www.beren.nl/welkom/    Restaurant              $               3.5   

   Google_Reviews__c Service_Options__c  \
0               2847            dine_in   

                               Payment_Options__c     Street    City  \
0  Credit cards, Debit cards, NFC mobile payments  Forum 101  Almere   

  PostalCode      Country  Lead_Enriched__c         Date_Enriched_At__c  
0    1315 TD  Netherlands              True  2026-04-24T23:36:15.961979  


In [5]:
# Update Salesforce
if not preview_df.empty:
    records = preview_df.to_dict('records')
    for r in records: r.pop('Name', None)
    try:
        results = sf.bulk.Lead.update(records)
        successes = [r for r in results if r["success"]]
        failures = [r for r in results if not r["success"]]
        logger.info(f"Update complete: {len(successes)} successful, {len(failures)} failed")
        if failures:
            for i, f in enumerate(failures[:5]):
                lead_id = records[results.index(f)]['Id']
                errors = f.get('errors', [])
                logger.error(f"Lead {lead_id} failed: {errors[0].get('message') if errors else 'Unknown error'}")
    except Exception as e:
        logger.error(f"Bulk update failed: {e}")
else:
    logger.info("No data to update")

INFO: Update complete: 1 successful, 0 failed
